# Credit Scoring with TensorFlow/Keras + Feast Feature Store

This notebook demonstrates a **TensorFlow/Keras** deep learning model for credit scoring using Feast as the feature platform.

## What's different from PyTorch notebook
- Keras `Sequential` API with EarlyStopping callback
- Training history visualization
- TF-Serving compatible model export
- `tf.data.Dataset` pipeline for batch training

## Cell 1: Install Dependencies

In [ ]:
!pip install -q tensorflow feast scikit-learn pandas matplotlib

## Cell 2: Load Data and Initialize Feast

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from feast import FeatureStore

print(f"TensorFlow version: {tf.__version__}")

fs = FeatureStore(repo_path="../feature_repo")
loans = pd.read_parquet("../feature_repo/data/loan_table.parquet")

print(f"Loan records: {len(loans)}")
loans.head(3)

## Cell 3: Retrieve Training Features from Feast (Point-in-Time Correct)

In [ ]:
feast_features = [
    "zipcode_features:city", "zipcode_features:state", "zipcode_features:location_type",
    "zipcode_features:tax_returns_filed", "zipcode_features:population",
    "zipcode_features:total_wages",
    "credit_history:credit_card_due", "credit_history:mortgage_due",
    "credit_history:student_loan_due", "credit_history:vehicle_loan_due",
    "credit_history:hard_pulls", "credit_history:missed_payments_2y",
    "credit_history:missed_payments_1y", "credit_history:missed_payments_6m",
    "credit_history:bankruptcies",
    "total_debt_calc:total_debt_due",
]

training_df = fs.get_historical_features(
    entity_df=loans, features=feast_features
).to_df()

print(f"Training shape: {training_df.shape}")
training_df.describe()

## Cell 4: Preprocess and Build tf.data Pipeline

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split

categorical_features = ["person_home_ownership", "loan_intent", "city", "state", "location_type"]
target = "loan_status"

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
scaler = StandardScaler()

df = training_df.dropna().copy()
encoder.fit(df[categorical_features])
df[categorical_features] = encoder.transform(df[categorical_features])

drop_cols = [target, "event_timestamp", "created_timestamp", "loan_id", "zipcode", "dob_ssn"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.reindex(sorted(X.columns), axis=1).fillna(0)
y = df[target].values

X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Build tf.data dataset for efficient batching
BATCH_SIZE = 32
train_ds = tf.data.Dataset.from_tensor_slices((X_train.astype(np.float32), y_train.astype(np.float32)))
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_test.astype(np.float32), y_test.astype(np.float32)))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Input dim: {X_train.shape[1]}")
print(f"Train batches: {len(train_ds)}, Val batches: {len(val_ds)}")

## Cell 5: Build and Train Keras Model

In [ ]:
input_dim = X_train.shape[1]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(input_dim,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", patience=5,
        restore_best_weights=True, mode="max"
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    verbose=1,
)

print(f"\n✅ Best val AUC: {max(history.history['val_auc']):.4f}")

## Cell 6: Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["auc"], label="Train")
axes[1].plot(history.history["val_auc"], label="Validation")
axes[1].set_title("AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("tf_training_history.png", dpi=150)
plt.show()

## Cell 7: Evaluate + Save Model (TF-Serving Compatible)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

probs = model.predict(X_test.astype(np.float32), verbose=0).flatten()
preds = (probs >= 0.5).astype(int)

print(f"AUC-ROC: {roc_auc_score(y_test, probs):.4f}")
print()
print(classification_report(y_test, preds, target_names=["Rejected", "Approved"]))

# Save in SavedModel format (compatible with TF-Serving)
model.save("tf_credit_model")
print("\n✅ Model saved to tf_credit_model/ (TF-Serving compatible)")

## Cell 8: Online Inference via Feast

In [ ]:
# New loan application
application = {
    "zipcode": [76104],
    "dob_ssn": ["19630621_4278"],
    "loan_amnt": [20000],
    "person_home_ownership": ["OWN"],
    "loan_intent": ["HOME_IMPROVEMENT"],
}

# Retrieve from Feast online store
online_features = fs.get_online_features(
    entity_rows=[{
        "zipcode": application["zipcode"][0],
        "dob_ssn": application["dob_ssn"][0],
        "loan_amnt": application["loan_amnt"][0],
    }],
    features=feast_features,
).to_dict()

features = application.copy()
features.update(online_features)
df_infer = pd.DataFrame.from_dict(features)
df_infer[categorical_features] = encoder.transform(df_infer[categorical_features])
df_infer = df_infer.drop(columns=["zipcode", "dob_ssn"], errors="ignore")
df_infer = df_infer.reindex(sorted(df_infer.columns), axis=1).fillna(0)
X_infer = scaler.transform(df_infer).astype(np.float32)

prob = float(model.predict(X_infer, verbose=0)[0][0])
decision = "✅ APPROVED" if prob >= 0.5 else "❌ REJECTED"

print(f"Loan Decision:       {decision}")
print(f"Approval Probability: {prob:.4f}")